# REACT 2026 — Leakage-Safe Feature Pipeline

Thin wrapper over `src/`. Everything substantive lives in the modules so the
same code runs locally and on Kaggle, and so the reproducibility check has one
source of truth.

**On Kaggle:** attach this repo as a dataset (or `%pip install` nothing — the
pipeline needs only pandas / numpy / scipy / scikit-learn / lightgbm, plus
torch for the optional GNN tier) and set `SRC` below to the directory holding
`src/`.

## The three rules this pipeline is built around

1. **Strictly past-only.** Every feature for a transaction at time *t* uses
   only information from before *t*. Proven, not asserted — see the truncation
   test at the bottom.
2. **Combined-stream features.** Features are built over `concat(train, test)`
   sorted by time. The rules explicitly allow this ("a device's known
   transaction history can include test-period rows that occurred earlier than
   the row being scored") and it is *required*: building on train alone would
   starve every test row of its within-test history.
3. **No random K-fold.** Validation is a 62-day forward block that reproduces
   the real train→test geometry exactly.

In [ ]:
import sys, time

SRC = ".."  # directory containing src/ ; on Kaggle e.g. "/kaggle/input/react2026-src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import numpy as np
import pandas as pd

from src import config as C
from src import leakage_checks as LC, validation as V
from src.io_utils import get_stream
from src.features import amount, encoding, entity, graph, temporal, velocity

pd.set_option("display.width", 200)
print("target encoding fit cutoff:", C.TRAIN_END)
print("signup-inconsistency feature enabled:", C.USE_SIGNUP_INCONSISTENCY)

## 1. Load and clean

Missing `merchant_category` / `device_type` / `location` (~0.4–0.6%) are kept
as an explicit `__NA__` level rather than mode-imputed: missingness here is
MCAR (NA-row fraud rates 0.0199 / 0.0193 / 0.0170 against a 0.0176 base), so
imputing would erase the "field was absent" fact and buy nothing.

In [ ]:
df = get_stream()
print(f"{len(df):,} rows  ({(~df.is_test).sum():,} train / {df.is_test.sum():,} test)")
print(f"fraud rate: {df.fraud.mean():.4%}")
df.head(3)

## 2. Build features

| block | what it captures |
|---|---|
| `temporal` | hour/day, cyclical encodings, night flag, per-customer circular rhythm |
| `amount` | amount vs the customer's own history; drift-robust trailing percentile ranks |
| `velocity` | customer / device / merchant recency and rolling counts (1h→168h) |
| `entity` | novelty of customer↔entity pairs, device sharing, diversity, location movement |
| `encoding` | past-only smoothed target encoding, frozen at the train cutoff |
| `graph` | snapshot bipartite graph: degrees, components, spectral + GNN embeddings |

In [ ]:
USE_GNN = True  # Tier C; set False for a ~30s faster build

parts, blocks = [], {}
for name, fn in [
    ("temporal", temporal.build),
    ("amount", amount.build),
    ("velocity", velocity.build),
    ("entity", entity.build),
    ("graph", lambda d: graph.build(d, use_gnn=USE_GNN, verbose=False)),
]:
    t = time.time()
    blk = fn(df)
    blocks[name] = list(blk.columns)
    parts.append(blk)
    print(f"  {name:9s} {blk.shape[1]:3d} cols  {time.time()-t:6.1f}s")

base = pd.concat(parts, axis=1)
print(f"\n{base.shape[1]} label-free features")

## 3. Validation

The real task is a **62-day forward block** immediately after the cutoff, so
`primary_62d` reproduces that shape and is the only fold used for selection.
The 30-day walk-forward folds measure stability.

**Target encoding is rebuilt per fold** with that fold's cutoff. Reusing the
production encoder (fitted to 2026-07-15) inside a fold ending 2026-05-14 would
feed validation-period labels into the encoder and inflate the score.

In [ ]:
print(V.describe_folds(df).to_string(index=False))

In [ ]:
import lightgbm as lgb

PARAMS = dict(
    objective="binary", metric="average_precision", learning_rate=0.05,
    num_leaves=64, min_data_in_leaf=100, feature_fraction=0.7,
    bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0,
    num_threads=0, verbosity=-1, seed=C.SEED,
)

ts, is_test = df[C.TIME_COL], df["is_test"].to_numpy()
y = df[C.TARGET].to_numpy(dtype=float)
rows = []

for fold in V.all_folds():
    te = encoding.build(df, fit_cutoff=fold.train_end)   # per-fold encoder
    X = pd.concat([base, te], axis=1)
    tm, vm = fold.train_mask(ts, is_test), fold.val_mask(ts, is_test)
    model = lgb.train(
        PARAMS, lgb.Dataset(X[tm], label=y[tm]), num_boost_round=1500,
        valid_sets=[lgb.Dataset(X[vm], label=y[vm])],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
    )
    p = model.predict(X[vm], num_iteration=model.best_iteration)
    a = V.ap(y[vm], p)
    rows.append({"fold": fold.name, "val_ap": a, "x_base": a / y[vm].mean(),
                 "best_iter": model.best_iteration})
    print(f"  {fold.name:12s} val_AP={a:.4f} ({a/y[vm].mean():5.1f}x base)")

res = pd.DataFrame(rows)
print(f"\nmean {res.val_ap.mean():.4f}  std {res.val_ap.std():.4f}")

## 4. Leakage checks

The truncation test is the decisive one: if a feature at time *t* depended on
anything after *t*, deleting the future would change it. Recomputing the whole
pipeline on a truncated stream and diffing proves the past-only property
directly, rather than relying on code review.

Anything derived per customer must live in a feature module, **not** in the
loader — otherwise it runs upstream of this test and escapes it.

In [ ]:
def build_label_free(d):
    return pd.concat(
        [temporal.build(d), amount.build(d), velocity.build(d), entity.build(d)], axis=1
    )

LC.check_no_banned_columns(base)
print("banned columns .............. PASS")

ap_rep = LC.check_no_label_in_features(base, y)
print(f"no near-perfect feature ..... PASS (max single-feature AP {ap_rep.ap.max():.4f})")

LC.check_truncation_invariance(build_label_free, df, pd.Timestamp("2026-05-01"))
print("truncation invariance ....... PASS")

boundary = LC.check_boundary_continuity(base, df)
print(f"train/test boundary ......... {(boundary.flag=='REVIEW').sum()} flagged")
boundary[boundary.flag == "REVIEW"]

## 5. Write the processed matrices

Modelling — hyperparameters, ensembling, calibration — happens downstream from
here. The LightGBM run above exists only to confirm the split behaves and to
rank the feature blocks.

In [ ]:
X_full = pd.concat([base, encoding.build(df, fit_cutoff=C.TRAIN_END)], axis=1)
keys = df[[C.ID_COL, C.TIME_COL]].reset_index(drop=True)

train_out = pd.concat([keys[~is_test], X_full[~is_test], df.loc[~is_test, [C.TARGET]]], axis=1)
test_out = pd.concat([keys[is_test], X_full[is_test]], axis=1)
train_out.to_parquet(C.PROCESSED / "train_features.parquet", index=False)
test_out.to_parquet(C.PROCESSED / "test_features.parquet", index=False)
print(train_out.shape, test_out.shape)

## 6. Modelling

Config search on the primary fold, then a greedy hill-climb blend.

**Do not equal-weight across model families here.** XGBoost (0.7123) and
CatBoost (0.7088) both score below every LightGBM config (0.7191-0.7239), so a
naive rank-average lands at 0.7214 — *worse* than simply using the best single
model. The hill-climb picks weights from evidence and excluded both non-LGB
families outright.

Blending is on **ranks**: average precision depends only on ordering, which is
also why there is no calibration step.

In [ ]:
from train_model import stage_search, stage_blend, stage_final, load_base

base = load_base(df)          # cached label-free matrix
stage_search(df, base)        # ~15 min: 7 configs on the 62-day fold
spec = stage_blend(df)        # choose blend weights from saved val predictions
spec

### Final fit and submission

Refits the chosen members on all labelled data (3 seeds each) with round counts
from the fold scaled by 1.2, then writes `submission.csv`.

The target encoder is refit at `TRAIN_END` for this stage — the production
setting that real test rows will see.

In [ ]:
stage_final(df, base, n_seeds=3)   # ~45 min, writes submission.csv